# DeepAgents + MongoDB Atlas — Travel Planner Walkthrough

This notebook demonstrates every MongoDB Atlas integration in the demo using the **CLI path** (`uv run deepagents-mongodb plan`). Each section invokes the agent as a subprocess — exactly the way you would run it from a terminal — then inspects MongoDB directly to show what was persisted.

## Persona: **niko**
| Field | Value |
|---|---|
| Origin | San Francisco (SFO) |
| Diet | Pescatarian |
| Hotel preference | Boutique / Ryokan |
| Budget | $5,000 total for 2 travelers |
| Thread 1 | Tokyo |
| Thread 2 | Seoul |

## Architecture

```
  uv run deepagents-mongodb plan --user niko --query "..."
          |
          v
  build_travel_agent()  ←  MongoDBSaver checkpointer
          |
          +---> Orchestrator  ←  MongoDBStore (preferences / past_trips)
          |         |
          |         +---> destination_researcher  ←  Vector Search (destinations_kb)
          |         +---> flight_researcher       ←  flights collection
          |         +---> hotel_researcher        ←  hotels collection
          |         +---> activity_planner        ←  activities collection
          |         +---> restaurant_recommender  ←  restaurants collection
          |         +---> itinerary_compiler
          |
          +---> SessionScopedSemanticCache  ←  semantic_cache + Atlas ANN
```

## What each section covers

| Section | MongoDB feature | Teaching point |
|---|---|---|
| 1 | Setup & bootstrap | Collections, indexes, seed data |
| 2 | CLI cold run | All 6 subagents, MISS+STORE cache log |
| 3 | Subagent tree from checkpoint | State lives in MongoDB, not Python memory |
| 4 | `MongoDBSaver` short-term memory | New process, same thread_id → conversation resumes |
| 5 | `MongoDBStore` long-term memory | Preferences + past trips recalled in Thread 2 |
| 6 | `SessionScopedSemanticCache` | MISS → STORE → HIT (paraphrase), cross-session isolation |
| 7 | Cleanup | Reset demo |

> Run cells top-to-bottom. Thread 1 (Tokyo) takes 40–60 s cold; warm runs are 5–15 s.

## 0. Environment setup

In [1]:
import sys, pathlib, subprocess, re, time, os
from pymongo import MongoClient

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))

from deepagents_mongodb.config import load_settings
settings = load_settings()
print('DB          :', settings.mongo.db)
print('Atlas host  :', settings.mongo.uri.split('@')[-1].split('/')[0])
print('Collections :', settings.mongo.checkpoints, '/', settings.mongo.agent_store,
      '/', settings.mongo.semantic_cache)

DB          : travel_planner
Atlas host  : ai-demo.ebx0sm5.mongodb.net
Collections : checkpoints / agent_store / semantic_cache


## 1. Atlas bootstrap

Creates all collections, operational indexes, Atlas Vector Search indexes, and seeds the knowledge base. **Idempotent** — safe to re-run.

In [ ]:
from scripts import bootstrap_atlas
bootstrap_atlas.main()

In [4]:
# Verify collection counts after bootstrap
raw_client = MongoClient(settings.mongo.uri)
db_handle = raw_client[settings.mongo.db]

for coll in ('flights', 'hotels', 'activities', 'restaurants', 'destinations_kb'):
    n = db_handle[coll].count_documents({})
    print(f'  {coll:20s}: {n:5d} docs')

  flights             :  3090 docs
  hotels              :   176 docs
  activities          :    70 docs
  restaurants         :    90 docs
  destinations_kb     :    17 docs


## 2. CLI cold run — Thread 1: Tokyo

We invoke the CLI as a **subprocess** — exactly what you run in a terminal. This exercises:
- All 6 subagents (destination, flights, hotels, activities, restaurants, itinerary)
- `MongoDBSaver` checkpointer (every step persisted)
- `MongoDBStore` long-term memory (preferences written at end)
- `SessionScopedSemanticCache` MISS + STORE

Persona constraints that map to existing seed data:
- **Origin SFO → Tokyo**: direct flights in DB (`ZH`, `UA`, `DL`, `NH`, `JL`, `KE` …)
- **Pescatarian**: `Sushi Saito`, `Tsukiji Outer Market Sushi`, `Hamacho Sushi Nakamura`, `Tempura Kondo`
- **Boutique/Ryokan**: `Tokyo Boutique 1/2` ($240–$255), `Tokyo Ryokan 1/2` ($320–$335)
- **$5 k total for 2**: round-trip ~$960–$1,400/person + 5-night boutique ~$1,275 + food/activities fits

In [2]:
USER_ID   = 'niko'
QUERY_1   = (
    # 'Plan a 5-day Tokyo trip in July 2026 for two pescatarians. '
    # 'Budget $5000 total. Origin: Hong Kong. '
    # 'Prefer boutique or ryokan accommodation.'
    '请规划一份2026年7月的5天东京双人游行程。两位旅客皆为海鲜素食者(不吃肉，但吃海鲜).'
    '总预算为$5000美元。出发地: 香港。住宿偏好精品酒店或日式旅馆。'
)

# Build the CLI command.  We run the module directly so the notebook
# venv is used regardless of how `uv` resolves the entry-point.
PYTHON = sys.executable
ENV    = {**os.environ, 'PYTHONUNBUFFERED': '1'}

cmd_cold = [
    PYTHON, '-m', 'deepagents_mongodb.cli', 'plan',
    '--user',  USER_ID,
    '--query', QUERY_1,
]

print('Running:', ' '.join(cmd_cold[:6]), '...')
t0 = time.time()
result_cold = subprocess.run(
    cmd_cold, capture_output=True, text=True, env=ENV, cwd=str(ROOT)
)
elapsed_cold = time.time() - t0

# ── Capture thread_id from CLI output ───────────────────────────────────────
m = re.search(r'thread-niko-[a-f0-9]+', result_cold.stdout)
thread_id_1 = m.group(0) if m else None

# ── Print the Rich output (last 4000 chars keeps the itinerary) ──────────────
print(result_cold.stdout[-4000:])

# ── Semantic cache log lines come on stderr (logging module) ─────────────────
cache_lines_cold = [l for l in result_cold.stderr.splitlines() if 'SemanticCache' in l]
if cache_lines_cold:
    print('\n=== SemanticCache activity (cold run) ===')
    for l in cache_lines_cold:
        print(' ', l)

print(f'\nelapsed_cold = {elapsed_cold:.1f}s')
print(f'thread_id_1  = {thread_id_1}')

Running: /Users/nikofeng/Documents/Github/deepagents-mongodb/.venv/bin/python -m deepagents_mongodb.cli plan --user niko ...
*Weather note:** Keep this day flexible in case of jet lag or rain.

### Day 2 — Asakusa / Senso-ji
**Theme:** classic Tokyo culture with compact sightseeing.

- **Morning:** Head to **Asakusa** early to beat heat and crowds. Visit 
**Senso-ji**, Nakamise shopping street, and nearby temple grounds.
- **Lunch:** **Tsukiji Outer Market Sushi** if you want seafood-forward lunch, 
or choose a lighter meal around Asakusa.
- **Afternoon:** Optional riverfront break or indoor café stop to escape 
humidity.
- **Evening:** Return to hotel for rest, then dinner at **T's Tantan Vegan 
Ramen** for a meat-free, easy dinner.
- **Transit tip:** Use the IC card for all local rides; keep transfers short in 
rainy weather.

### Day 3 — Tsukiji Outer Market + Ginza
**Theme:** food-focused day with shopping and refined dining.

- **Early morning:** Visit **Tsukiji Outer Market** for

## 3. Subagent delegation tree — from MongoDB

The orchestrator's full state — messages, todos, virtual files — was written to the `checkpoints` collection after **every** LangGraph step. We now read it back directly from MongoDB using `pymongo`, without any live agent object. This proves the state lives in the database, not in Python memory.

In [5]:
# ── Raw checkpoint document ──────────────────────────────────────────────────
n_checkpoints = db_handle[settings.mongo.checkpoints].count_documents(
    {'thread_id': thread_id_1}
)
print(f'Checkpoints stored for thread {thread_id_1}: {n_checkpoints}')

latest_doc = db_handle[settings.mongo.checkpoints].find_one(
    {'thread_id': thread_id_1},
    sort=[('checkpoint_id', -1)]
)
print('Document keys:', sorted(k for k in latest_doc.keys() if k != '_id'))
print('checkpoint_id:', latest_doc.get('checkpoint_id'))
print('type         :', latest_doc.get('type'))

Checkpoints stored for thread thread-niko-5f905620: 98
Document keys: ['checkpoint', 'checkpoint_id', 'checkpoint_ns', 'metadata', 'parent_checkpoint_id', 'thread_id', 'type']
checkpoint_id: 1f15ca3d-9e75-6474-8012-9a2b0354f083
type         : msgpack


In [6]:
# ── Decode checkpoint → extract messages → build delegation tree ─────────────
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer

serde  = JsonPlusSerializer()
state  = serde.loads_typed((latest_doc['type'], latest_doc['checkpoint']))
ch_val = state.get('channel_values', {})

# todos written by write_todos tool
todos = ch_val.get('todos') or []
print('=== Orchestrator Todo List ===\n')
if todos:
    for i, t in enumerate(todos, 1):
        content = t.get('content', t) if isinstance(t, dict) else t
        print(f'  {i}. {content}')
else:
    print('  (no todos found)')

# subagent task() calls from message tool_calls
messages = ch_val.get('messages', [])
subagent_calls = []
for msg in messages:
    for tc in (getattr(msg, 'tool_calls', None) or []):
        if tc.get('name') == 'task':
            args = tc.get('args', {})
            subagent_calls.append((
                args.get('subagent_type') or args.get('name', '?'),
                (args.get('description') or '')[:120]
            ))

print('\n=== Subagent Delegation Tree ===\n')
if subagent_calls:
    print('  Orchestrator')
    for name, desc in subagent_calls:
        print(f'  \u2514\u2500 {name}')
        print(f'       "{desc}..."')
else:
    print('  (no task() calls found in messages)')

=== Orchestrator Todo List ===

  1. Recall traveler preferences and past trips
  2. Research Tokyo destination details
  3. Research flights from Hong Kong to Tokyo for July 2026
  4. Research boutique hotel or ryokan options in Tokyo
  5. Plan activities for a 5-day Tokyo trip for two
  6. Recommend seafood-vegetarian-friendly restaurants in Tokyo
  7. Compile final itinerary and budget into itinerary.md
  8. Save stable traveler preferences and past-trip summary to memory

=== Subagent Delegation Tree ===

  (no task() calls found in messages)


In [7]:
# ── Render itinerary.md read back from the checkpoint ────────────────────────
from IPython.display import Markdown

files = ch_val.get('files', {})
itinerary = files.get('/itinerary.md') or files.get('itinerary.md')
if isinstance(itinerary, dict):
    itinerary = itinerary.get('content', '')

Markdown(itinerary or '_No itinerary.md found in checkpoint — check CLI output above._')

_No itinerary.md found in checkpoint — check CLI output above._

## 4. Short-term memory — `MongoDBSaver` checkpointer

The `checkpoints` collection stores the **complete** LangGraph state after every step (messages, virtual files, todos). A brand-new process that presents the same `thread_id` picks up exactly where the previous one left off — there is zero in-memory dependency.

We prove this by spawning a **second subprocess** with `--thread-id` pointing at Thread 1 and asking a stateless follow-up question. The agent can only answer it by reading the persisted message history from MongoDB.

In [ ]:
cmd_followup = [
    PYTHON, '-m', 'deepagents_mongodb.cli', 'plan',
    '--user',      USER_ID,
    '--query',     'What was the total budget you proposed, and which hotel did you recommend?',
    '--thread-id', thread_id_1,   # resume the same thread
    '--no-cache',                 # isolate this demo from cache effects
]

print(f'Resuming thread {thread_id_1} in a new process...')
result_followup = subprocess.run(
    cmd_followup, capture_output=True, text=True, env=ENV, cwd=str(ROOT)
)
print(result_followup.stdout[-2000:])
# Expected: agent answers with the exact budget + hotel from the original itinerary
# despite being a completely fresh process with no in-memory state.

## 5. Long-term memory — `MongoDBStore`

`MongoDBStore` persists **cross-session** knowledge: traveler preferences and past-trip summaries. It is shared across all threads and all processes. The orchestrator writes to it with `save_traveler_preference` and `save_past_trip` at the end of each plan. Any future session can semantically search it with `recall_traveler_preferences` and `recall_past_trips`.

### 5a. Inspect what Thread 1 wrote about niko

In [ ]:
from deepagents_mongodb.store import ns_preferences, ns_past_trips
from deepagents_mongodb.agent import build_travel_agent

# Build a lightweight bundle just to access the store (no cache needed here)
bundle = build_travel_agent(settings, enable_cache=False)

print("=== Niko's preferences (from Thread 1) ===\n")
prefs = bundle.store.search(ns_preferences(USER_ID), query='diet hotel budget', limit=10)
if prefs:
    for r in prefs:
        print(' •', r.value.get('content'))
else:
    print('  (none yet — Thread 1 may not have saved preferences)')

print("\n=== Niko's past trips (from Thread 1) ===\n")
trips = bundle.store.search(ns_past_trips(USER_ID), query='Tokyo', limit=5)
if trips:
    for r in trips:
        print(' •', r.value.get('content'))
else:
    print('  (none yet)')

### 5b. Thread 2 — Seoul: cross-session preference recall

A new thread for a different destination. The orchestrator calls `recall_traveler_preferences` at the start, reads the pescatarian + boutique + $5k preferences saved by Thread 1, and uses them to tailor the Seoul plan — without the user repeating themselves.

**Seed data used:**
- Flights: `SFO → ICN` (Korean Air, United, Delta in DB)
- Restaurants: `Mingles`, `Noryangjin Fish Market`, `Haenyeo Restaurant` (all pescatarian)
- Hotels: Seoul Boutique, Seoul Business (in DB)

In [ ]:
QUERY_2 = (
    'Plan a 4-day Seoul trip in June 2026 for two travelers. '
    'Same dietary preferences as before. Budget around $4000 total. '
    'Origin: San Francisco.'
)

cmd_seoul = [
    PYTHON, '-m', 'deepagents_mongodb.cli', 'plan',
    '--user',  USER_ID,
    '--query', QUERY_2,
]

print('Running Thread 2 (Seoul)...')
t0 = time.time()
result_seoul = subprocess.run(
    cmd_seoul, capture_output=True, text=True, env=ENV, cwd=str(ROOT)
)
elapsed_seoul = time.time() - t0

m2 = re.search(r'thread-niko-[a-f0-9]+', result_seoul.stdout)
thread_id_2 = m2.group(0) if m2 else None

print(result_seoul.stdout[-3000:])
print(f'\nelapsed = {elapsed_seoul:.1f}s  thread_id_2 = {thread_id_2}')

In [ ]:
# ── Verify both trips are now in the store ────────────────────────────────────
print("=== Niko's past trips (after Thread 2) ===\n")
trips_after = bundle.store.search(ns_past_trips(USER_ID), query='trips', limit=10)
for r in trips_after:
    print(' •', r.value.get('content'))
# Expected: Tokyo trip (from Thread 1) + Seoul trip (from Thread 2)

print("\n=== Semantic recall: 'I like boutique hotels' ===\n")
for r in bundle.store.search(ns_preferences(USER_ID), query='boutique hotels', limit=3):
    print(' •', r.value.get('content'))

## 6. Semantic LLM cache — `SessionScopedSemanticCache`

### Design

| Property | Value |
|---|---|
| Cache key | Last user message only (plain text, not full conversation) |
| Scope | `session_id = thread_id` — each thread has its own isolated bucket |
| Similarity threshold | 0.92 cosine (Voyage-4 embeddings, 1024-dim) |
| Atlas ANN delay | ~3–5 s on M0 before a newly written vector is queryable |
| Cross-session | A new `thread_id` is **always a MISS** (isolation is a feature) |

The cache is useful when the same user asks semantically equivalent follow-ups within one conversation (e.g. "best time to visit Tokyo" → "which month is good for Tokyo").

### 6a. Cold run — MISS + STORE

In [ ]:
QUERY_CACHE = 'What is the best time of year to visit Tokyo for cherry blossoms?'

cmd_cache_cold = [
    PYTHON, '-m', 'deepagents_mongodb.cli', 'plan',
    '--user',  USER_ID,
    '--query', QUERY_CACHE,
]

print('Cold run (expect MISS + STORE)...')
t0 = time.time()
result_cache_cold = subprocess.run(
    cmd_cache_cold, capture_output=True, text=True, env=ENV, cwd=str(ROOT)
)
elapsed_cache_cold = time.time() - t0

mc = re.search(r'thread-niko-[a-f0-9]+', result_cache_cold.stdout)
thread_id_cache = mc.group(0) if mc else None

# Show cache log lines
cache_cold_lines = [l for l in result_cache_cold.stderr.splitlines() if 'SemanticCache' in l]
print('\n=== SemanticCache activity ===')
for l in cache_cold_lines:
    print(' ', l)

print(f'\nelapsed_cache_cold = {elapsed_cache_cold:.1f}s')
print(f'thread_id_cache    = {thread_id_cache}')

# How many docs are in the cache collection now?
# Note: CLI calls clear_session() on exit, so count may be 0.
# We check BEFORE the next run, when the new thread's session is still live.
n_cache = db_handle[settings.mongo.semantic_cache].count_documents({})
print(f'\nsemantic_cache docs in Atlas: {n_cache}')
print('(0 = CLI already called clear_session; see log line above)')

### 6b. Warm run — paraphrase HIT (same `thread_id`)

We re-use **the same `thread_id`** so `session_id` matches. The paraphrase embeds to a very similar vector (cosine ~0.94–0.97), triggering a HIT. We wait 5 s first to give Atlas ANN time to index the new vector.

In [ ]:
print('Waiting 5 s for Atlas ANN to index the stored vector...')
time.sleep(5)

QUERY_CACHE_PARAPHRASE = 'When should I visit Tokyo to see the cherry blossoms in bloom?'

cmd_cache_warm = [
    PYTHON, '-m', 'deepagents_mongodb.cli', 'plan',
    '--user',      USER_ID,
    '--query',     QUERY_CACHE_PARAPHRASE,
    '--thread-id', thread_id_cache,   # same session → same session_id filter
]

print('Warm run (expect HIT)...')
t0 = time.time()
result_cache_warm = subprocess.run(
    cmd_cache_warm, capture_output=True, text=True, env=ENV, cwd=str(ROOT)
)
elapsed_cache_warm = time.time() - t0

cache_warm_lines = [l for l in result_cache_warm.stderr.splitlines() if 'SemanticCache' in l]
print('\n=== SemanticCache activity (warm run) ===')
for l in cache_warm_lines:
    print(' ', l)

print(f'\nelapsed_cache_cold = {elapsed_cache_cold:.1f}s')
print(f'elapsed_cache_warm = {elapsed_cache_warm:.1f}s')
if elapsed_cache_warm > 0:
    print(f'speedup            = {elapsed_cache_cold / elapsed_cache_warm:.1f}x')

### 6c. Cross-session isolation — new `thread_id` is always a MISS

The cache is **session-scoped**: each `thread_id` has its own `session_id` filter in the Atlas vector index. A new thread cannot hit another thread's cache entries. This is intentional — it prevents answers from leaking across independent conversations.

In [ ]:
# New subprocess with NO --thread-id → fresh thread_id generated → new session_id
cmd_new_session = [
    PYTHON, '-m', 'deepagents_mongodb.cli', 'plan',
    '--user',  USER_ID,
    '--query', QUERY_CACHE_PARAPHRASE,   # exact same paraphrase as the warm run
    # no --thread-id → brand-new thread
]

print('New session run (expect MISS despite same question)...')
t0 = time.time()
result_new_session = subprocess.run(
    cmd_new_session, capture_output=True, text=True, env=ENV, cwd=str(ROOT)
)
elapsed_new_session = time.time() - t0

new_session_cache = [l for l in result_new_session.stderr.splitlines() if 'SemanticCache' in l]
print('\n=== SemanticCache activity (new session) ===')
for l in new_session_cache:
    print(' ', l)
print(f'\nelapsed = {elapsed_new_session:.1f}s')
print('\nExpected: SemanticCache MISS — cross-session isolation confirmed.')

### 6d. Inspect the `semantic_cache` collection directly

In [ ]:
cache_coll = db_handle[settings.mongo.semantic_cache]
total = cache_coll.count_documents({})
print(f'Documents in semantic_cache: {total}')
print('(CLI calls clear_session() on exit, so count is typically 0 here)\n')

for doc in cache_coll.find({}, {'session_id': 1, 'text': 1, '_id': 0}).limit(10):
    print(' •', {'session_id': doc.get('session_id'), 'text': (doc.get('text') or '')[:80]})

## 7. Cleanup

Wipes all collections (checkpoints, store, cache, seed data) while keeping all Atlas indexes in place. Run `bootstrap_atlas.main()` to re-seed.

In [ ]:
bundle.close()
raw_client.close()

from scripts import reset_demo
reset_demo.main()
print('Done. Run bootstrap_atlas.main() to re-seed.')